# Lecture 16: Logistic Regression

This notebook models a binary campaign response outcome and evaluates probability predictions.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
campaign = pd.read_csv(DATA / "campaign_response.csv")
campaign["responded"].mean()


In [ ]:
logit = smf.logit(
    "responded ~ visits_last_month + emails_opened + discount_pct + prior_spend_eur + C(segment)",
    data=campaign,
).fit()
print(logit.summary())


In [ ]:
odds_ratios = np.exp(logit.params)
odds_ratios.sort_values(ascending=False)


In [ ]:
campaign = campaign.assign(predicted_probability=logit.predict(campaign))
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=campaign, x="predicted_probability", hue="responded", bins=20, ax=ax)
ax.set(title="Predicted response probabilities")


In [ ]:
threshold = 0.5
predicted_class = (campaign["predicted_probability"] >= threshold).astype(int)
cm = confusion_matrix(campaign["responded"], predicted_class)
print(classification_report(campaign["responded"], predicted_class))
print(f"ROC AUC: {roc_auc_score(campaign['responded'], campaign['predicted_probability']):.3f}")
ConfusionMatrixDisplay(cm).plot()


## LLM Check

Give an LLM a confusion matrix and ask it to compute precision and recall. Verify the arithmetic before using the explanation.
